In [1]:
print("hello inference")

hello inference


In [13]:
!pip install wandb -q
import wandb
api = wandb.Api()

In [15]:
artifact = api.artifact("ikvas22-free-university-of-tbilisi/Facial_Expression_Recognition/model-deep-cnn-adam-lr0.0001-drop0.25:latest")
artifact_dir = artifact.download()

print(f"Model downloaded to: {artifact_dir}")

wandb:   1 of 1 files downloaded.  


Model downloaded to: /content/artifacts/model-deep-cnn-adam-lr0.0001-drop0.25:v0


In [16]:
# ============================================================
# IMPORTS
# ============================================================
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cuda


In [17]:
class DeepCNN(nn.Module):
    def __init__(self, dropout_rate=0.25):
        super(DeepCNN, self).__init__()

        self.conv_block1 = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(dropout_rate)
        )

        self.conv_block2 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(dropout_rate)
        )

        self.conv_block3 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(dropout_rate)
        )

        self.conv_block4 = nn.Sequential(
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(dropout_rate)
        )

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512 * 3 * 3, 1024),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(128, 7)
        )

    def forward(self, x):
        x = self.conv_block1(x)
        x = self.conv_block2(x)
        x = self.conv_block3(x)
        x = self.conv_block4(x)
        x = self.fc(x)
        return x

In [18]:
import os

model = DeepCNN(dropout_rate=0.25).to(device)
model.load_state_dict(
    torch.load(os.path.join(artifact_dir, "best_model.pth"), map_location=device)
)
model.eval()
print("Model loaded from W&B!")

Model loaded from W&B!


In [19]:
emotion_labels = {
    0: "Angry", 1: "Disgust", 2: "Fear",
    3: "Happy", 4: "Sad", 5: "Surprise", 6: "Neutral"
}

TEST_PATH = "/content/test.csv" # depends on the directory
test_df = pd.read_csv(TEST_PATH)

def predict_batch(model, test_df, batch_size=64):
    all_predictions = []

    X_test = np.array([
        list(map(int, row.split()))
        for row in test_df['pixels']
    ]).reshape(-1, 1, 48, 48) / 255.0

    for i in range(0, len(X_test), batch_size):
        batch = X_test[i:i+batch_size]
        tensor = torch.tensor(batch, dtype=torch.float32).to(device)

        with torch.no_grad():
            outputs = model(tensor)
            predicted = outputs.argmax(dim=1).cpu().numpy()
            all_predictions.extend(predicted)

        if i % 500 == 0:
            print(f"Processed {i}/{len(X_test)} samples...")

    return all_predictions

predictions = predict_batch(model, test_df)
print(f"Total predictions: {len(predictions)}")

Processed 0/7178 samples...
Total predictions: 7178


In [20]:
submission = pd.DataFrame({
    'emotion': predictions
})
submission.index.name = 'id'
submission.to_csv('submission.csv')
print("Submission saved!")
print(submission['emotion'].value_counts().sort_index())

Submission saved!
emotion
0     790
1      89
2     496
3    1721
4    1305
5     904
6    1873
Name: count, dtype: int64
